# เปรียบเทียบ YOLOv8 vs CNN — ตรวจจับวัตถุดิบอาหาร

**Dataset:** `food-ingredients-dataset` v4 จาก Roboflow Universe (120 คลาส, YOLO format, CC BY 4.0)

โน้ตบุ๊กนี้ทำ 3 อย่าง ตามข้อ 1.4.3.1–1.4.3.2 ของโครงงาน:
1. เทรน **YOLOv8n** (fine-tune จาก pretrained weights) บนชุดข้อมูลจริง แล้ววัด **mAP@0.5** และ **mAP@0.5:0.95**
2. เทรน **CNN classifier** (ResNet18 transfer learning) บนชุดข้อมูลเดียวกัน (ครอปจาก bounding box) แล้ววัด **accuracy / precision / recall / F1**
3. สรุปตารางเปรียบเทียบทั้งสองโมเดล — ตัวเลขทั้งหมดมาจากการเทรน/ทดสอบจริงในโน้ตบุ๊กนี้ ไม่มีค่าใดถูกใส่มาเอง

**วิธีใช้:** เปิดใน Google Colab → ตั้ง Runtime เป็น GPU (Runtime > Change runtime type > T4 GPU) → รันทีละเซลล์ตามลำดับ

## 0. ติดตั้งไลบรารีที่ใช้

In [ ]:
!pip install -q ultralytics roboflow torchmetrics scikit-learn

## 1. โหลดชุดข้อมูลจาก Roboflow

ใส่ API key ของคุณเอง (หาได้จาก Roboflow > Settings > API Keys) — **ห้าม commit key นี้ขึ้น GitHub**

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "ใส่_API_KEY_ของคุณตรงนี้"  # TODO: แก้เป็น key จริง

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("food-recipe-ingredient-images-0gnku").project("food-ingredients-dataset")
dataset = project.version(4).download("yolov8")

DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

In [ ]:
# ตรวจสอบ data.yaml ที่ Roboflow สร้างให้ (path, จำนวนคลาส, ชื่อคลาส)
import yaml

with open(f"{DATASET_DIR}/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

print("จำนวนคลาส:", data_cfg["nc"])
print("ตัวอย่างคลาส:", data_cfg["names"][:15])

### (ตัวเลือก) ใช้ทั้ง 120 คลาส vs ใช้ subset เพื่อลองก่อน

การเทรนทั้ง 120 คลาสด้วย GPU ฟรีของ Colab ใช้เวลานาน (หลายชั่วโมงต่อรอบ) ถ้าอยากเทสต์ pipeline ให้รันจบเร็วก่อน
ให้ตั้ง `USE_SUBSET = True` เพื่อเทรนแค่บางคลาสก่อน แล้วค่อยเปลี่ยนเป็น `False` ตอนรันจริงเพื่อเอาผลไปเขียนรายงาน

In [ ]:
USE_SUBSET = True  # ตั้งเป็น False เมื่อพร้อมเทรนเต็มทั้ง 120 คลาสสำหรับผลจริงในรายงาน
SUBSET_CLASSES = ["Tomato", "Garlic", "Onion", "Chicken", "Egg", "Carrot", "Cucumber", "Chili Pepper -Khursani-"]

print("โหมด:", "subset (" + str(len(SUBSET_CLASSES)) + " คลาส)" if USE_SUBSET else "เต็ม 120 คลาส")

## 2. เทรน YOLOv8n (fine-tune จาก pretrained weights)

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")  # เริ่มจาก pretrained COCO weights แล้ว fine-tune ต่อ

yolo_results = yolo_model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=50,          # เพิ่มเป็น 100+ ถ้าต้องการผลแม่นยำขึ้นสำหรับรายงานจริง
    imgsz=640,
    batch=16,
    patience=15,        # early stopping ถ้าไม่ดีขึ้น
    project="runs_yolo",
    name="ingredient_detector",
)

### 2.1 วัดผล YOLO บน test set — mAP@0.5, mAP@0.5:0.95, precision, recall

ตัวเลขในเซลล์นี้มาจาก `model.val()` ที่รันจริงบน test set ที่โมเดลไม่เคยเห็นตอนเทรน

In [ ]:
best_yolo = YOLO("runs_yolo/ingredient_detector/weights/best.pt")

yolo_metrics = best_yolo.val(data=f"{DATASET_DIR}/data.yaml", split="test")

print(f"mAP@0.5      : {yolo_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {yolo_metrics.box.map:.4f}")
print(f"Precision    : {yolo_metrics.box.mp:.4f}")
print(f"Recall       : {yolo_metrics.box.mr:.4f}")

## 3. สร้างชุดข้อมูลสำหรับ CNN (ครอปรูปตาม bounding box เดิม)

YOLO ใช้รูปเต็ม + bounding box แต่ CNN classifier ต้องการรูปที่ครอปแล้วต่อ 1 label — โค้ดนี้แปลง YOLO-format labels
ให้เป็นโฟลเดอร์ ImageFolder (1 โฟลเดอร์ต่อคลาส) โดยครอปแต่ละ bounding box ออกมาเป็นรูปแยก
เพื่อให้ CNN เทรน/ทดสอบบน **วัตถุดิบชิ้นเดียวกันกับที่ YOLO ใช้** — การเปรียบเทียบจึงยุติธรรม

In [ ]:
import os
from pathlib import Path
from PIL import Image

def yolo_to_classification(dataset_dir, split, class_names, out_dir, subset=None):
    img_dir = Path(dataset_dir) / split / "images"
    lbl_dir = Path(dataset_dir) / split / "labels"
    out_split_dir = Path(out_dir) / split

    count = 0
    for img_path in img_dir.glob("*.*"):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            continue
        w, h = image.size

        for i, line in enumerate(lbl_path.read_text().strip().splitlines()):
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            cls_name = class_names[cls_id]
            if subset and cls_name not in subset:
                continue

            xc, yc, bw, bh = [float(v) for v in parts[1:5]]
            x1 = max(0, int((xc - bw / 2) * w))
            y1 = max(0, int((yc - bh / 2) * h))
            x2 = min(w, int((xc + bw / 2) * w))
            y2 = min(h, int((yc + bh / 2) * h))
            if x2 <= x1 or y2 <= y1:
                continue

            crop = image.crop((x1, y1, x2, y2))
            class_dir = out_split_dir / cls_name
            class_dir.mkdir(parents=True, exist_ok=True)
            crop.save(class_dir / f"{img_path.stem}_{i}.jpg")
            count += 1
    return count

CNN_DATA_DIR = "cnn_dataset"
subset = SUBSET_CLASSES if USE_SUBSET else None

for split in ["train", "valid", "test"]:
    n = yolo_to_classification(DATASET_DIR, split, data_cfg["names"], CNN_DATA_DIR, subset=subset)
    print(f"{split}: สร้างรูปครอปแล้ว {n} รูป")

## 4. เทรน CNN classifier (ResNet18 transfer learning)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ใช้:", device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f"{CNN_DATA_DIR}/train", transform=transform)
valid_ds = datasets.ImageFolder(f"{CNN_DATA_DIR}/valid", transform=transform)
test_ds = datasets.ImageFolder(f"{CNN_DATA_DIR}/test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=32)
test_loader = DataLoader(test_ds, batch_size=32)

num_classes = len(train_ds.classes)
print("จำนวนคลาสที่ใช้เทรน CNN:", num_classes)

cnn_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, num_classes)
cnn_model = cnn_model.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=1e-4)

EPOCHS = 15  # เพิ่มได้ถ้าต้องการความแม่นยำสูงขึ้นสำหรับรายงานจริง

for epoch in range(EPOCHS):
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} — loss: {running_loss/total:.4f}, train acc: {train_acc:.4f}")

### 4.1 วัดผล CNN บน test set — accuracy, precision, recall, F1

ตัวเลขมาจากการรันจริงบน test set เดียวกันกับที่ใช้วัด YOLO ด้านบน (คนละ representation แต่ข้อมูลชุดเดียวกัน)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

cnn_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = cnn_model(images)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cnn_acc = accuracy_score(all_labels, all_preds)
cnn_p, cnn_r, cnn_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

print(f"Accuracy  : {cnn_acc:.4f}")
print(f"Precision : {cnn_p:.4f}")
print(f"Recall    : {cnn_r:.4f}")
print(f"F1-score  : {cnn_f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=train_ds.classes, zero_division=0))

## 5. สรุปเปรียบเทียบ YOLOv8 vs CNN

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "โมเดล": ["YOLOv8n (fine-tuned)", "ResNet18 (transfer learning)"],
    "งานที่ทำ": ["Object detection (ตำแหน่ง+คลาส)", "Image classification (คลาสอย่างเดียว)"],
    "metric หลัก": ["mAP@0.5", "Accuracy"],
    "ผลลัพธ์": [f"{yolo_metrics.box.map50:.4f}", f"{cnn_acc:.4f}"],
    "Precision": [f"{yolo_metrics.box.mp:.4f}", f"{cnn_p:.4f}"],
    "Recall": [f"{yolo_metrics.box.mr:.4f}", f"{cnn_r:.4f}"],
})
comparison

### หมายเหตุสำหรับเขียนรายงาน

- **YOLOv8** บอกได้ทั้ง "มีอะไร" และ "อยู่ตรงไหนในรูป" (bounding box) เหมาะกับกรณีที่รูปมีวัตถุดิบหลายอย่างพร้อมกัน (สถานการณ์จริงของแอปนี้ — ถ่ายรูปในตู้เย็นที่มีวัตถุดิบหลายอย่าง)
- **CNN classifier** บอกได้แค่ "รูปนี้คือวัตถุดิบอะไร" ทีละ 1 อย่างต่อรูป (ต้องครอปมาก่อน) เหมาะกับกรณีถ่ายรูปวัตถุดิบทีละอย่างชัดๆ
- ตัวเลขในตารางนี้เป็นผลจากการเทรน `EPOCHS` ตามที่ตั้งไว้ข้างต้น — ถ้าต้องการผลที่ดีขึ้นสำหรับรายงานฉบับสมบูรณ์ ให้เพิ่มจำนวน epoch และตั้ง `USE_SUBSET = False` เพื่อใช้ทั้ง 120 คลาส แล้วรันใหม่